# WAF Assessment Tool — Uninstall

Tears down everything `install.ipynb` deployed, by reading the install manifest written at install time.

**What this removes:** Genie space, Lakeview dashboard, Databricks App, reload job, app workspace files, `waf_cache` schema (CASCADE), and the catalog if install created it.

**Requires:** install was performed by a version of `install.ipynb` that writes `_install_manifest`. For older installs, delete resources manually.

**Failure model:** this script fails loudly. If a resource is already gone, comment out that line and re-run.

In [ ]:
# STEP 1: CONFIGURATION
catalog    = "<<your_catalog>>"   # the catalog used during install
install_id = None                  # None = uninstall the most recent install; else set the install_id string


In [ ]:
# STEP 2: LOAD MANIFEST ROW
import requests

ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
api_url = ctx.apiUrl().get()
token   = ctx.apiToken().get()

where = "" if install_id is None else f"WHERE install_id = '{install_id}'"
row = spark.sql(
    f"SELECT * FROM `{catalog}`.`waf_cache`.`_install_manifest` {where} "
    f"ORDER BY created_at DESC LIMIT 1"
).collect()[0]

print(f"Uninstalling install_id={row.install_id} (created {row.created_at})")
for f in ["catalog", "catalog_created_by_install", "workspace_path", "app_name",
          "job_id", "dashboard_id", "genie_space_id"]:
    print(f"  {f:30s}: {row[f]}")

In [ ]:
# STEP 3: DELETE API-BACKED RESOURCES
headers = {"Authorization": f"Bearer {token}"}

if row.genie_space_id:
    print(" Deleting Genie space...")
    requests.delete(f"{api_url}/api/2.0/genie/spaces/{row.genie_space_id}", headers=headers).raise_for_status()

print(" Deleting Lakeview dashboard...")
requests.delete(f"{api_url}/api/2.0/lakeview/dashboards/{row.dashboard_id}", headers=headers).raise_for_status()

print(" Deleting Databricks App...")
requests.delete(f"{api_url}/api/2.0/apps/{row.app_name}", headers=headers).raise_for_status()

print(" Deleting reload job...")
requests.post(
    f"{api_url}/api/2.1/jobs/delete",
    headers={**headers, "Content-Type": "application/json"},
    json={"job_id": row.job_id},
).raise_for_status()

print(" Deleting app workspace path...")
requests.post(
    f"{api_url}/api/2.0/workspace/delete",
    headers={**headers, "Content-Type": "application/json"},
    json={"path": row.workspace_path, "recursive": True},
).raise_for_status()

In [ ]:
# STEP 4: DROP UC OBJECTS
print(f" Dropping schema {row.catalog}.waf_cache (CASCADE)...")
spark.sql(f"DROP SCHEMA `{row.catalog}`.`waf_cache` CASCADE")

if row.catalog_created_by_install:
    print(f" Dropping catalog {row.catalog} (created by install)...")
    spark.sql(f"DROP CATALOG `{row.catalog}`")
else:
    print(f" Catalog {row.catalog} pre-existed → kept.")

print("\n Uninstall complete.")